In [1]:
# Run this only if keras-tuner is not installed
# !pip install keras-tuner
#!pip install tensorflow pandas numpy matplotlib scikit-learn keras-tuner tensorboard

In [2]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras_tuner as kt

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("TensorFlow version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

TensorFlow version: 2.21.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [3]:
# ============================================================
# 2. PROJECT SETTINGS - DINNER RUN VERSION
# ============================================================

MAX_WORDS = 10000
MAX_LEN = 200

BATCH_SIZE = 64
TUNING_EPOCHS = 5
FINAL_EPOCHS = 20
VALIDATION_SPLIT = 0.2
MAX_TRIALS = 10

MODEL_DIR = "saved_models"
RESULTS_DIR = "results"
TUNER_DIR = "tuner_results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TUNER_DIR, exist_ok=True)

print("Settings loaded successfully.")

Settings loaded successfully.


In [4]:
# ============================================================
# 3. LOAD IMDB DATASET
# ============================================================

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_WORDS)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))
print("Example sequence:", x_train[0][:20])
print("Example label:", y_train[0])

Training samples: 25000
Testing samples: 25000
Example sequence: [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
Example label: 1


In [5]:
# ============================================================
# 4. PAD SEQUENCES
# ============================================================

x_train_pad = pad_sequences(
    x_train,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

x_test_pad = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Padded training shape:", x_train_pad.shape)
print("Padded testing shape:", x_test_pad.shape)

Padded training shape: (25000, 200)
Padded testing shape: (25000, 200)


In [6]:
# ============================================================
# 5. BUILD LSTM MODEL FOR RANDOM SEARCH
# ============================================================

def build_lstm_tuner_model(hp):
    model = Sequential()

    model.add(tf.keras.Input(shape=(MAX_LEN,)))

    model.add(Embedding(
        input_dim=MAX_WORDS,
        output_dim=hp.Choice("embedding_dim", values=[32, 64, 128])
    ))

    model.add(LSTM(
        units=hp.Choice("lstm_units", values=[32, 64, 128])
    ))

    model.add(Dropout(
        rate=hp.Choice("dropout_rate", values=[0.2, 0.3, 0.5])
    ))

    model.add(Dense(1, activation="sigmoid"))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.001, 0.0005, 0.0001]
    )

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [7]:
# ============================================================
# 6. RANDOM SEARCH FOR LSTM
# ============================================================

lstm_tuner = kt.RandomSearch(
    build_lstm_tuner_model,
    objective="val_accuracy",
    max_trials=MAX_TRIALS,
    executions_per_trial=1,
    directory=TUNER_DIR,
    project_name="imdb_lstm_random_search",
    overwrite=True
)

lstm_tuner.search(
    x_train_pad,
    y_train,
    validation_split=VALIDATION_SPLIT,
    epochs=TUNING_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

best_lstm_hp = lstm_tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best LSTM Hyperparameters")
print("Embedding dim:", best_lstm_hp.get("embedding_dim"))
print("LSTM units:", best_lstm_hp.get("lstm_units"))
print("Dropout:", best_lstm_hp.get("dropout_rate"))
print("Learning rate:", best_lstm_hp.get("learning_rate"))

lstm_tuner.results_summary(num_trials=MAX_TRIALS)

Trial 10 Complete [00h 02m 23s]
val_accuracy: 0.8600000143051147

Best val_accuracy So Far: 0.8691999912261963
Total elapsed time: 00h 16m 57s
Best LSTM Hyperparameters
Embedding dim: 128
LSTM units: 64
Dropout: 0.5
Learning rate: 0.0001
Results summary
Results in tuner_results\imdb_lstm_random_search
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 04 summary
Hyperparameters:
embedding_dim: 128
lstm_units: 64
dropout_rate: 0.5
learning_rate: 0.0001
Score: 0.8691999912261963

Trial 05 summary
Hyperparameters:
embedding_dim: 128
lstm_units: 128
dropout_rate: 0.2
learning_rate: 0.0001
Score: 0.8640000224113464

Trial 09 summary
Hyperparameters:
embedding_dim: 32
lstm_units: 128
dropout_rate: 0.2
learning_rate: 0.0001
Score: 0.8600000143051147

Trial 02 summary
Hyperparameters:
embedding_dim: 128
lstm_units: 32
dropout_rate: 0.3
learning_rate: 0.0001
Score: 0.8208000063896179

Trial 01 summary
Hyperparameters:
embedding_dim: 128
lstm_units: 32
dropout_rate: 0.

In [8]:
# ============================================================
# 7. BUILD GRU MODEL FOR RANDOM SEARCH
# ============================================================

def build_gru_tuner_model(hp):
    model = Sequential()

    model.add(tf.keras.Input(shape=(MAX_LEN,)))

    model.add(Embedding(
        input_dim=MAX_WORDS,
        output_dim=hp.Choice("embedding_dim", values=[32, 64, 128])
    ))

    model.add(GRU(
        units=hp.Choice("gru_units", values=[32, 64, 128])
    ))

    model.add(Dropout(
        rate=hp.Choice("dropout_rate", values=[0.2, 0.3, 0.5])
    ))

    model.add(Dense(1, activation="sigmoid"))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.001, 0.0005, 0.0001]
    )

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [9]:
# ============================================================
# 8. RANDOM SEARCH FOR GRU
# ============================================================

gru_tuner = kt.RandomSearch(
    build_gru_tuner_model,
    objective="val_accuracy",
    max_trials=MAX_TRIALS,
    executions_per_trial=1,
    directory=TUNER_DIR,
    project_name="imdb_gru_random_search",
    overwrite=True
)

gru_tuner.search(
    x_train_pad,
    y_train,
    validation_split=VALIDATION_SPLIT,
    epochs=TUNING_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

best_gru_hp = gru_tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best GRU Hyperparameters")
print("Embedding dim:", best_gru_hp.get("embedding_dim"))
print("GRU units:", best_gru_hp.get("gru_units"))
print("Dropout:", best_gru_hp.get("dropout_rate"))
print("Learning rate:", best_gru_hp.get("learning_rate"))

gru_tuner.results_summary(num_trials=MAX_TRIALS)

Trial 10 Complete [00h 02m 24s]
val_accuracy: 0.8550000190734863

Best val_accuracy So Far: 0.8691999912261963
Total elapsed time: 00h 24m 48s
Best GRU Hyperparameters
Embedding dim: 64
GRU units: 64
Dropout: 0.3
Learning rate: 0.001
Results summary
Results in tuner_results\imdb_gru_random_search
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 01 summary
Hyperparameters:
embedding_dim: 64
gru_units: 64
dropout_rate: 0.3
learning_rate: 0.001
Score: 0.8691999912261963

Trial 06 summary
Hyperparameters:
embedding_dim: 128
gru_units: 32
dropout_rate: 0.5
learning_rate: 0.001
Score: 0.8673999905586243

Trial 02 summary
Hyperparameters:
embedding_dim: 64
gru_units: 64
dropout_rate: 0.2
learning_rate: 0.001
Score: 0.8659999966621399

Trial 04 summary
Hyperparameters:
embedding_dim: 32
gru_units: 128
dropout_rate: 0.5
learning_rate: 0.001
Score: 0.8596000075340271

Trial 07 summary
Hyperparameters:
embedding_dim: 128
gru_units: 64
dropout_rate: 0.2
learning_rate: 

In [10]:
# ============================================================
# 9. TRAIN FINAL LSTM MODEL USING BEST HYPERPARAMETERS
# ============================================================


best_lstm_model = lstm_tuner.hypermodel.build(best_lstm_hp)

lstm_model_path = os.path.join(MODEL_DIR, "lstm_imdb_model.keras")

lstm_checkpoint = ModelCheckpoint(
    lstm_model_path,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

lstm_history = best_lstm_model.fit(
    x_train_pad,
    y_train,
    validation_split=VALIDATION_SPLIT,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[lstm_checkpoint],
    verbose=1
)

print("Saved best LSTM model to:", lstm_model_path)

Epoch 1/20
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5144 - loss: 0.6929
Epoch 1: val_accuracy improved from None to 0.50520, saving model to saved_models\lstm_imdb_model.keras

Epoch 1: finished saving model to saved_models\lstm_imdb_model.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 18s 53ms/step - accuracy: 0.5103 - loss: 0.6930 - val_accuracy: 0.5052 - val_loss: 0.6927
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5262 - loss: 0.6923
Epoch 2: val_accuracy improved from 0.50520 to 0.52160, saving model to saved_models\lstm_imdb_model.keras

Epoch 2: finished saving model to saved_models\lstm_imdb_model.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - accuracy: 0.5246 - loss: 0.6921 - val_accuracy: 0.5216 - val_loss: 0.6919
Epoch 3/20
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5533 - loss: 0.6876
Epoch 3: val_accuracy improved from 0.52160 to 0.74360, saving model to saved_models\lstm_imdb_model.keras

Epoch 3: finished saving model to saved_mo

In [11]:
# ============================================================
# 10. TRAIN FINAL GRU MODEL USING BEST HYPERPARAMETERS
# ============================================================

best_gru_model = gru_tuner.hypermodel.build(best_gru_hp)

gru_model_path = os.path.join(MODEL_DIR, "gru_imdb_model.keras")

gru_checkpoint = ModelCheckpoint(
    gru_model_path,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

gru_history = best_gru_model.fit(
    x_train_pad,
    y_train,
    validation_split=VALIDATION_SPLIT,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[gru_checkpoint],
    verbose=1
)

print("Saved best GRU model to:", gru_model_path)

Epoch 1/20
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.5100 - loss: 0.6933
Epoch 1: val_accuracy improved from None to 0.52420, saving model to saved_models\gru_imdb_model.keras

Epoch 1: finished saving model to saved_models\gru_imdb_model.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 49ms/step - accuracy: 0.5170 - loss: 0.6925 - val_accuracy: 0.5242 - val_loss: 0.6897
Epoch 2/20
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.5773 - loss: 0.6711
Epoch 2: val_accuracy did not improve from 0.52420
313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 48ms/step - accuracy: 0.5793 - loss: 0.6691 - val_accuracy: 0.5136 - val_loss: 0.6851
Epoch 3/20
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.6685 - loss: 0.6042
Epoch 3: val_accuracy improved from 0.52420 to 0.72860, saving model to saved_models\gru_imdb_model.keras

Epoch 3: finished saving model to saved_models\gru_imdb_model.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 48ms/step - accuracy: 0.7106 - loss: 0.5729 - val_accuracy: 0.7286 - v